# Notebook 3 - Gráficos

Este notebook reúne toda a visualização do projeto, partindo da base limpa gerada pelo notebook 1. São produzidos três blocos de gráficos:

1. Boxplots de apoio à detecção de outliers, que justificam visualmente o agrupamento adotado no notebook 1.

2. Gráficos de relação entre variáveis: Temperature por Run Case, Temperature versus Active Limit e Temperature por Key Area.

3. Histogramas por região térmica, correspondentes ao item (d) do Artefato 5.

Pré-requisito: executar 01_leitura.ipynb antes deste notebook.

## Setup do ambiente

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S1_S2"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
%pip install seaborn

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

DIR_SAIDA = "dados_intermediarios"
CAMINHO_LIMPO = os.path.join(DIR_SAIDA, "df_limpo.csv")

if not os.path.exists(CAMINHO_LIMPO):
    raise FileNotFoundError(
        "df_limpo.csv não encontrado. Execute o notebook 01_leitura.ipynb antes deste."
    )

df_limpo = pd.read_csv(CAMINHO_LIMPO)
df_limpo["Possui Active Limit"] = df_limpo["Possui Active Limit"].astype(bool)

print("Base carregada:", df_limpo.shape)

## Boxplots de apoio à detecção de outliers

O boxplot da temperatura sozinho não ajuda na identificação dos outliers, já que reúne peças diferentes, que recebem quantidades de calor diferentes. Ao separar por peça, a visualização fica muito mais informativa e evidencia por que a detecção precisou ser agrupada por sensor, e não aplicada sobre a coluna inteira.

In [ ]:
# Apenas fazer o boxplot da temperatura não ajuda na identificação,
# já que são peças diferentes que recebem quantidades de calor diferentes.
df_limpo.boxplot(column="Temperature")
plt.show()

# Ao separar por peça, fica mais fácil a visualização.
df_limpo.boxplot(column="Temperature", by="TC Name", figsize=(20, 9), rot=90)
plt.show()

## Gráficos de relação de variáveis

Os gráficos a seguir demonstram as relações entre as variáveis do dataframe. A visualização permite que os dados tratados sejam analisados de forma mais simples pelo grupo de desenvolvimento e auxilia na tomada de decisões quanto à organização dos passos seguintes. Nesta seção, três gráficos foram elaborados, sendo o primeiro deles a relação entre Temperature e Run Case.

### Temperature x Run Case

O gráfico mostra que há diferença nas medições dos diferentes sensores, correspondentes às peças do fogão, em um mesmo teste de função. A parte majoritária das temperaturas encontra-se abaixo dos 100 °C, o que se deve ao fato de que os outliers correspondem a peças expostas à cavidade de aquecimento do fogão, em número reduzido na comparação com o todo, que apresenta temperaturas mais controladas.

In [ ]:
df_limpo.boxplot(column="Temperature", by="Run Case", figsize=(10, 6), rot=90)
plt.show()

### Temperature vs Active Limit

O gráfico de dispersão relaciona a temperatura medida em cada sensor com o limite térmico ativo (Active Limit) daquele componente, mostrando o quão próxima cada leitura está do ponto de falha. Foram considerados apenas os sensores com limite de segurança real, já que sensores de monitoramento sem limite associado foram excluídos dessa análise específica.

Duas linhas de referência guiam a leitura: a vermelha (y = x) marca a falha absoluta, e a laranja (y = 0,9x) marca a margem de segurança de 10% exigida pela engenharia.

Pontos entre as duas linhas indicam componentes que passam no teste, mas operam fora da margem de segurança recomendada, ajudando a priorizar onde o modelo e a engenharia devem focar atenção. Pontos acima da linha vermelha correspondem a leituras que ultrapassaram o limite, entre as quais estão as do sensor Oven RTD Connector Body, cuja anomalia foi registrada no notebook 1.

In [ ]:
# Filtrar a base: mantém apenas sensores que possuem um Active Limit real definido,
# excluindo os sensores de monitoramento sem limite de segurança associado
# (conforme confirmado pela Whirlpool e sinalizado na coluna "Possui Active Limit")
df_plot = df_limpo[df_limpo["Possui Active Limit"]]

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_plot, x="Active Limit", y="Temperature", alpha=0.6, color="#1f77b4")

max_limit = df_plot["Active Limit"].max()

# Linha de Falha (Y = X)
plt.plot([0, max_limit], [0, max_limit], color="red", linestyle="--",
         label="Falha Absoluta (Limite Real)")

# Linha de Segurança 10% (Y = 0.9X)
plt.plot([0, max_limit], [0, max_limit * 0.9], color="orange", linestyle="-", linewidth=2,
         label="Margem de Segurança (10%)")

plt.title("Temperatura Medida vs Limite de Segurança dos Componentes", fontsize=14, pad=15)
plt.xlabel("Limite Ativo Exigido (°C)", fontsize=12)
plt.ylabel("Temperatura Real Medida (°C)", fontsize=12)
plt.legend()
plt.grid(True, linestyle=":", alpha=0.5)
plt.show()

### Temperature vs Key Area

O boxplot compara a distribuição de temperatura entre as diferentes áreas do produto (Key Area), evidenciando quais regiões concentram os maiores riscos térmicos e a variabilidade das leituras em cada uma.

Cada caixa representa a distribuição de uma Key Area, com a linha central indicando a mediana e os pontos fora dela sinalizando outliers.

Áreas com caixas mais altas ou mais dispersas indicam componentes sob temperaturas mais extremas ou menos previsíveis, sendo bons candidatos a atenção prioritária tanto na engenharia física quanto no treinamento do modelo.

In [ ]:
df_limpo.boxplot(column="Temperature", by="Key Area", figsize=(12, 7), rot=45)
plt.title("Distribuição de temperatura por área do produto")
plt.suptitle("")
plt.xlabel("Key Area")
plt.ylabel("Temperature (°C)")
plt.show()

## Histogramas da Temperature por região térmica

Como a análise trabalha com uma única variável quantitativa (Temperature), a análise visual é feita por segmentação por região térmica do fogão, onde cada histograma reúne todas as leituras de temperatura dos termopares de uma mesma região, ao longo de todas as runs de simulação.

A coluna Key Area da base identifica a região à qual cada termopar pertence. A base contém exatamente sete regiões, o que define os sete histogramas. A contagem de termopares de cada região é impressa pela célula seguinte, a partir da própria base, em vez de fixada no texto.

Escala: cada histograma usa escala própria no eixo X.

In [ ]:
# Ordem de exibição e rótulos legíveis das regiões (mesma segmentação do notebook 2)
REGIOES = [
    "Mainback",
    "Coil Temps",
    "Main Vent Sys",
    "IPC Cooling",
    "IPC Comp",
    "Oven HMI",
    "CT Knobs",
]

ROTULOS_REGIAO = {
    "Mainback": "Mainback - peças da parede traseira",
    "Coil Temps": "Coil Temps - bobinas do cooktop",
    "Main Vent Sys": "Main Vent Sys - ventilação principal",
    "IPC Cooling": "IPC Cooling - ventilação cooktop/IPC",
    "IPC Comp": "IPC Comp - componentes da placa IPC",
    "Oven HMI": "Oven HMI - interface do forno",
    "CT Knobs": "CT Knobs - encoders do cooktop",
}

regioes = df_limpo["Key Area"].dropna().unique().tolist()
faltantes = [r for r in regioes if r not in REGIOES]
assert not faltantes, f"Regiões presentes na base e ausentes da lista: {faltantes}"

print(f"Regiões encontradas: {len(regioes)}\n")
for regiao in REGIOES:
    sub = df_limpo[df_limpo["Key Area"] == regiao]
    limites = sorted(sub["Active Limit"].dropna().unique())
    print(f"  {regiao:<15} | n = {len(sub):>4} | termopares: {sub['TC Name'].nunique():>2} | "
          f"limites distintos: {len(limites)}")

In [ ]:
# Geração dos histogramas
BINS = "sturges"

for regiao in REGIOES:
    sub = df_limpo[df_limpo["Key Area"] == regiao]
    serie = sub["Temperature"].dropna()
    limites = sorted(sub["Active Limit"].dropna().unique())

    figura, eixo = plt.subplots(figsize=(8, 4.5))
    eixo.hist(serie, bins=BINS, edgecolor="black", color="#4C72B0")
    eixo.axvline(serie.mean(), color="#55A868", linestyle="-", linewidth=1.8,
                 label=f"Média = {serie.mean():.1f} °C")
    eixo.axvline(serie.median(), color="#8172B2", linestyle=":", linewidth=1.8,
                 label=f"Mediana = {serie.median():.1f} °C")

    # Linha de limite apenas quando a região tem um único Active Limit
    if len(limites) == 1:
        eixo.axvline(limites[0], color="#C44E52", linestyle="--", linewidth=2,
                     label=f"Active Limit = {limites[0]:.0f} °C")
    else:
        eixo.plot([], [], " ", label=f"{len(limites)} Active Limits distintos na região")

    eixo.set_title(
        f"Distribuição da temperatura - {ROTULOS_REGIAO[regiao]}\n"
        f"(n = {len(serie)} leituras; {sub['TC Name'].nunique()} termopares)", fontsize=11)
    eixo.set_xlabel("Temperatura (°C)")
    eixo.set_ylabel("Frequência (nº de leituras)")
    eixo.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# Painel das 7 regiões
figura, eixos = plt.subplots(4, 2, figsize=(13, 15))
eixos = eixos.flatten()

for indice, regiao in enumerate(REGIOES):
    eixo = eixos[indice]
    sub = df_limpo[df_limpo["Key Area"] == regiao]
    serie = sub["Temperature"].dropna()
    limites = sorted(sub["Active Limit"].dropna().unique())

    eixo.hist(serie, bins=BINS, edgecolor="black", color="#4C72B0")
    eixo.axvline(serie.mean(), color="#55A868", linewidth=1.5, label=f"Média = {serie.mean():.1f}")
    eixo.axvline(serie.median(), color="#8172B2", linestyle=":", linewidth=1.5,
                 label=f"Mediana = {serie.median():.1f}")
    if len(limites) == 1:
        eixo.axvline(limites[0], color="#C44E52", linestyle="--", linewidth=1.8,
                     label=f"Limite = {limites[0]:.0f} °C")

    eixo.set_title(f"{ROTULOS_REGIAO[regiao]}\n(n = {len(serie)})", fontsize=9)
    eixo.set_xlabel("Temperatura (°C)", fontsize=8)
    eixo.set_ylabel("Frequência", fontsize=8)
    eixo.legend(fontsize=7)

eixos[-1].axis("off")
figura.suptitle("Distribuição da temperatura por região térmica", fontsize=13, y=1.0)
plt.tight_layout()
plt.show()

### Comentário

Os sete histogramas são, no conjunto, consistentes com o teste estatístico realizado no notebook 2. O Shapiro-Wilk rejeitou H₀ para a variável Temperature no conjunto completo, e as sete regiões também rejeitam H₀ individualmente.

O grau de reforço visual, porém, não é uniforme. Em cinco regiões (Mainback, Coil Temps, Main Vent Sys, IPC Cooling e CT Knobs) a forma reforça o teste de maneira inequívoca, por assimetria acentuada ou multimodalidade evidente. Em IPC Comp e Oven HMI a distribuição é próxima da normal e o reforço é apenas parcial, já que a rejeição de H₀ é amplificada pelo tamanho da amostra e o Shapiro-Wilk detecta desvios pequenos demais para terem consequência prática.

Duas causas se destacam. A primeira é a multimodalidade induzida pelo desenho experimental, pois as runs são configurações distintas de ventilador, housing, defletor e modo de operação, e cada patamar térmico resultante gera os vales visíveis em Main Vent Sys, IPC Cooling e Coil Temps. A segunda é a heterogeneidade de componentes dentro da mesma região, como em Mainback, onde os termopares têm limites normativos de 60 °C a 165 °C e a agregação de leituras tão distintas produz a cauda extrema observada.

Na prática, procedimentos que assumam normalidade não são apropriados para esta base, tanto na análise exploratória quanto na avaliação dos resíduos do modelo. Recomenda-se estatísticas robustas na caracterização das variáveis e testes não paramétricos nas comparações entre configurações, além de tratamento segmentado por Key Area, já que cada região tem escala térmica e estrutura de distribuição próprias.

## Histogramas antes e depois da normalização Min-Max

Cada figura reúne dois painéis da mesma amostra: à esquerda a escala original em graus Celsius, à direita a escala adimensional restrita ao intervalo de 0 a 1. Como a normalização é uma transformação linear, as alturas das barras são idênticas nos dois painéis; o que muda é a unidade do eixo horizontal.

A normalização é aplicada aqui com o mínimo e o máximo da própria região, e não da coluna inteira, de modo que cada painel direito ocupe integralmente o intervalo de 0 a 1 e a forma da distribuição fique legível. Cabe registrar que essa é uma escolha de visualização: a padronização efetivamente aplicada ao conjunto de treino, no notebook 2, é o escore-z sobre a coluna completa.

In [ ]:
from pathlib import Path

DESTINO = Path("../../documents/assets/histogramas/posEscalonamento")
DESTINO.mkdir(parents=True, exist_ok=True)

ARQUIVOS = {
    "Mainback": "mainback.png",
    "Coil Temps": "coilTemps.png",
    "Main Vent Sys": "mainVentSys.png",
    "IPC Cooling": "ipcCooling.png",
    "IPC Comp": "ipcComp.png",
    "Oven HMI": "ovenHmi.png",
    "CT Knobs": "ctKnobs.png",
}

for regiao in REGIOES:
    serie = df_limpo.loc[df_limpo["Key Area"] == regiao, "Temperature"]
    minimo, maximo = serie.min(), serie.max()
    normalizada = (serie - minimo) / (maximo - minimo)

    figura, (esquerda, direita) = plt.subplots(1, 2, figsize=(14.19, 4.9))
    paineis = (
        (esquerda, serie, "#4C72B0", r"$\bf{ANTES}$ - escala original",
         "Temperatura (°C)", 1, " °C"),
        (direita, normalizada, "#DD8452", r"$\bf{DEPOIS}$ - Min-Max [0, 1]",
         "Temperatura normalizada (adimensional)", 3, ""),
    )

    for eixo, dados, cor, titulo, rotulo_x, casas, sufixo in paineis:
        eixo.hist(dados, bins=BINS, edgecolor="black", color=cor)
        eixo.axvline(dados.mean(), color="#55A868", linewidth=1.8,
                     label=f"Média = {dados.mean():.{casas}f}{sufixo}")
        eixo.axvline(dados.median(), color="#8172B2", linestyle=":", linewidth=1.8,
                     label=f"Mediana = {dados.median():.{casas}f}{sufixo}")
        eixo.set_title(titulo, fontsize=11)
        eixo.set_xlabel(rotulo_x)
        eixo.set_ylabel("Frequência (nº de leituras)")
        eixo.legend(fontsize=8)

    figura.suptitle(f"{ROTULOS_REGIAO[regiao]} (n = {len(serie)})", fontsize=13)
    plt.tight_layout()
    figura.savefig(DESTINO / ARQUIVOS[regiao], dpi=100)
    plt.show()

print("Figuras gravadas em:", DESTINO.resolve())

### Comentário

A comparação lado a lado confirma que a normalização Min-Max não altera a forma da distribuição: a assimetria da Mainback, a bimodalidade da IPC Cooling e os patamares da Main Vent Sys permanecem exatamente como estavam, e apenas o eixo horizontal muda de unidade.

Disso decorre a conclusão relevante para o modelo: o escalonamento coloca as variáveis em escala comparável, mas não corrige a não normalidade constatada nos testes do notebook 2. As recomendações de estatística robusta e de testes não paramétricos, portanto, permanecem válidas depois da normalização.